## Question 1

In [ ]:
%pip install -q opencv-python-headless easyocr matplotlib pillow pyttsx3

In [ ]:
import re
import cv2
import matplotlib.pyplot as plt
import easyocr
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))
image = cv2.imread(image_path)

if image is None:
    raise ValueError("Invalid image file")

plt.figure(figsize=(9, 6))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
gray = cv2.resize(gray, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)
gray = cv2.bilateralFilter(gray, 9, 75, 75)
processed = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 11)

reader = easyocr.Reader(["en"], gpu=False)
results = reader.readtext(processed)
lines = [item[1] for item in results]
text = "\n".join(lines)

print(text)

In [ ]:
def extract_value(patterns, text):
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip(" :-")
    return "Not detected"

lines = [line.strip() for line in text.splitlines() if line.strip()]
name = lines[0] if lines else "Not detected"

information = {
    "Medicine name": name,
    "Strength": extract_value([r"(\\d+(?:\\.\\d+)?\\s?(?:mg|mcg|g|ml|%))"], text),
    "Dosage": extract_value([r"(?:dosage|dose|directions|take)\\s*[:\-]?\\s*(.{3,100})"], text),
    "Manufacturing date": extract_value([r"(?:mfg|manufacturing|manufactured)\\s*(?:date)?\\s*[:#\-]?\\s*([A-Z0-9/\-. ]{4,20})"], text),
    "Expiry date": extract_value([r"(?:exp|expiry|expires|use before)\\s*(?:date)?\\s*[:#\-]?\\s*([A-Z0-9/\-. ]{4,20})"], text),
    "Batch number": extract_value([r"(?:batch(?:\\s*(?:no|number))?|lot)\\s*[:#\-]?\\s*([A-Z0-9/\-]+)"], text),
    "MRP": extract_value([r"(?:mrp|price)\\s*[:#\-]?\\s*(₹?\\s?\\d+(?:\\.\\d{1,2})?)"], text)
}

for key, value in information.items():
    print(f"{key}: {value}")

In [ ]:
summary = ". ".join(f"{key}: {value}" for key, value in information.items())
print(summary)

import pyttsx3

engine = pyttsx3.init()
engine.setProperty("rate", 160)
engine.say(summary)
engine.runAndWait()

## Question 2

In [ ]:
%pip install -q opencv-python-headless deepface tf-keras matplotlib

In [ ]:
import cv2
import matplotlib.pyplot as plt
from google.colab import files
from deepface import DeepFace

uploaded = files.upload()
face_path = next(iter(uploaded))
face_image = cv2.imread(face_path)

if face_image is None:
    raise ValueError("Invalid image file")

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(face_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
analysis = DeepFace.analyze(
    img_path=face_path,
    actions=["emotion"],
    detector_backend="opencv",
    enforce_detection=True,
    silent=True
)

if isinstance(analysis, list):
    analysis = analysis[0]

emotion = analysis["dominant_emotion"]
scores = analysis["emotion"]

print("Detected emotion:", emotion)

responses = {
    "happy": "You look cheerful. Keep enjoying the moment!",
    "sad": "Take a gentle break if you need one.",
    "angry": "Take a slow breath and give yourself a moment to pause.",
    "surprise": "That looks like a surprising moment!",
    "fear": "Try grounding yourself and reaching out to someone you trust.",
    "disgust": "Take a moment to reset.",
    "neutral": "You appear calm. How can I help you?"
}

print("Assistant:", responses.get(emotion, "How can I help you?"))

In [ ]:
labels = list(scores.keys())
values = [scores[label] for label in labels]

plt.figure(figsize=(9, 4))
plt.bar(labels, values)
plt.xlabel("Emotion")
plt.ylabel("Score (%)")
plt.title("Facial Expression Detection")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
import pyttsx3

engine = pyttsx3.init()
engine.setProperty("rate", 160)
engine.say(responses.get(emotion, "How can I help you?"))
engine.runAndWait()